In [2]:
import cv2
import numpy as np
import math
import glob

In [3]:
image_files = glob.glob("site_data/bischwiller/captured_poses/192_168_1_11/*")
image_files.sort()
len(image_files)

8

In [4]:

# Paramètres connus de la caméra
HFOV_deg = 54.2  # champ de vision horizontal en degrés

# Chargement des images en niveaux de gris
images = [cv2.imread(fname, cv2.IMREAD_GRAYSCALE) for fname in image_files]
W = images[0].shape[1]  # largeur en pixels (supposée identique pour toutes les images)

# Calcul de la focale en pixels à partir du champ de vision horizontal
f = (W/2) / math.tan(math.radians(HFOV_deg/2))

# Fonction utilitaire pour convertir une coordonnée x (px) en angle horizontal (deg)
def x_to_angle(x_pixel):
    x_offset = x_pixel - W/2            # décalage par rapport au centre de l'image
    theta = math.degrees(math.atan(x_offset / f))
    return theta

# Initialisation du détecteur ORB et du matcher de descripteurs
orb = cv2.ORB_create(nfeatures=1000)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)  # crossCheck pour correspondances mutuelles:contentReference[oaicite:4]{index=4}

# Calcul des décalages angulaires entre images consécutives
angle_deltas = []  # liste des ΔΘ entre image i et i+1
for i in range(len(images) - 1):
    img1, img2 = images[i], images[i+1]
    # Détection et description des points clés
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)
    # Correspondance des descripteurs entre image i et i+1
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda m: m.distance)  # trier par qualité (distance Hamming)
    # Ne garder que les meilleures correspondances (par ex. 50 meilleures pour fiabiliser)
    good_matches = matches[:50]
    # Calculer les angles pour chaque paire correspondante et en déduire les angles de rotation
    delta_angles = []
    for m in good_matches:
        x1 = kp1[m.queryIdx].pt[0]  # coordonnée x du point dans img1
        x2 = kp2[m.trainIdx].pt[0]  # coordonnée x du point correspondant dans img2
        theta1 = x_to_angle(x1)
        theta2 = x_to_angle(x2)
        # Estimation du décalage de camera nécessaire pour que theta1 (dans img1) aligne theta2 (dans img2)
        delta = theta1 - theta2
        delta_angles.append(delta)
    # Moyenne (ou médiane) du décalage angulaire pour cette paire d'images
    if len(delta_angles) > 0:
        mean_delta = float(np.mean(delta_angles))
        angle_deltas.append(mean_delta)
        print(f"Rotation estimée de im_{i} vers im_{i+1}: {mean_delta:.2f} degrés")

# Utilisation de l'azimut absolu connu de l'image 3 pour calculer les autres
absolute_azimuths = [None] * len(images)
absolute_azimuths[3] = 348.0  # azimut absolu connu pour im_3
# En avant (images 4,5,6,...)
for j in range(3, len(images)-1):
    absolute_azimuths[j+1] = (absolute_azimuths[j] + angle_deltas[j]) % 360
# En arrière (images 2,1,0)
for j in range(3, 0, -1):
    absolute_azimuths[j-1] = (absolute_azimuths[j] - angle_deltas[j-1]) % 360

# Affichage des résultats
for idx, az in enumerate(absolute_azimuths):
    print(f"Azimut image {idx}: {az:.2f}°")


Rotation estimée de im_0 vers im_1: 7.91 degrés
Rotation estimée de im_1 vers im_2: -4.20 degrés
Rotation estimée de im_2 vers im_3: -11.63 degrés
Rotation estimée de im_3 vers im_4: 24.12 degrés
Rotation estimée de im_4 vers im_5: -3.80 degrés
Rotation estimée de im_5 vers im_6: 7.40 degrés
Rotation estimée de im_6 vers im_7: 0.41 degrés
Azimut image 0: 355.92°
Azimut image 1: 3.83°
Azimut image 2: 359.63°
Azimut image 3: 348.00°
Azimut image 4: 12.12°
Azimut image 5: 8.33°
Azimut image 6: 15.73°
Azimut image 7: 16.13°
